# Exposure compatibility audit

Audit purpose: decide whether recorded NCDB road-tractor casualty events and modeled tractor-trailer travel describe compatible populations. This notebook checks source transcription and numerator grain. It does **not** calculate collision rates, impute missing years, or validate model assumptions merely because its arithmetic passes.

Candidate VKT covers tractor-trailers over 15,000 kg; NCDB08 includes road tractors with or without semitrailers and has no weight field. Domestic-road versus registration-frame coverage, foreign travel and model calibration remain methodological gates. See `docs/exposure_compatibility_audit.md` for the controlling decision.

In [1]:
from pathlib import Path
import csv, hashlib, json
import duckdb
from IPython.display import display
root = Path.cwd()
if not (root/'data').exists(): root = root.parent
out = root/'reports/exposure'
out.mkdir(parents=True,exist_ok=True)
events_path = root/'data/derived/ncdb/collisions.parquet'
candidate_path = root/'data/reference/tractor_exposure_candidates.csv'
metadata_path = root/'data/reference/tractor_exposure_candidates.metadata.json'
assert candidate_path.exists() and metadata_path.exists()
candidates = list(csv.DictReader(candidate_path.open()))
metadata = json.loads(metadata_path.read_text())
display(candidates)
display(metadata)

[{'series_id': 'tc_2021_parliament_table10_tractor',
  'year': '2015',
  'vehicle_class': 'tractor_trailers',
  'minimum_weight_kg': '15000',
  'weight_operator': '>',
  'vkt_million_km': '34203.5',
  'unit': 'million_vehicle_kilometres',
  'estimate_type': 'econometric_forecast',
  'source_id': 'tc_commercial_vehicle_safety_2021',
  'source_table': 'Table 10',
  'eligibility_status': 'not_validated_for_ncdb08_rates'},
 {'series_id': 'tc_2021_parliament_table10_tractor',
  'year': '2016',
  'vehicle_class': 'tractor_trailers',
  'minimum_weight_kg': '15000',
  'weight_operator': '>',
  'vkt_million_km': '35898.0',
  'unit': 'million_vehicle_kilometres',
  'estimate_type': 'econometric_forecast',
  'source_id': 'tc_commercial_vehicle_safety_2021',
  'source_table': 'Table 10',
  'eligibility_status': 'not_validated_for_ncdb08_rates'},
 {'series_id': 'tc_2021_parliament_table10_tractor',
  'year': '2017',
  'vehicle_class': 'tractor_trailers',
  'minimum_weight_kg': '15000',
  'weight_op

{'audit_date': '2026-10-17',
 'candidate_csv': 'data/reference/tractor_exposure_candidates.csv',
 'candidate_csv_sha256': '3faa5533286a8ac39694d073bc9566ef9d31259daf4fd26235fa44871a78bf99',
 'series_id': 'tc_2021_parliament_table10_tractor',
 'title': 'Transport Canada modeled tractor-trailer VKT, Table 10, 2021 Annual Report to Parliament',
 'publisher': 'Transport Canada',
 'publication_month': '2026-05',
 'html_modified_date': '2026-06-25',
 'data_years': [2015, 2016, 2017, 2018, 2019, 2020, 2021],
 'grain': 'one calendar year and candidate exposure series',
 'unit': 'million_vehicle_kilometres',
 'conversion_to_vehicle_kilometres': 1000000,
 'source_table': 'Table 10',
 'pdf_physical_page': 57,
 'pdf_printed_page': 45,
 'methods_pdf_physical_page': 56,
 'methods_pdf_printed_page': 44,
 'extraction': 'Parsed exact tractor row of preserved HTML table with Python HTMLParser; checked all seven values against preserved PDF text. No graph digitization or extrapolation.',
 'published_prec

In [2]:
# Independent transcription check against the preserved source table.
from decimal import Decimal
from html.parser import HTMLParser
class Tables(HTMLParser):
    def __init__(self):
        super().__init__(); self.tables=[]; self.current=None; self.row=None; self.cell=None
    def handle_starttag(self,t,attrs):
        if t=='table': self.current=[]
        elif t=='tr' and self.current is not None: self.row=[]
        elif t in ('td','th') and self.row is not None: self.cell=[]
    def handle_data(self,data):
        if self.cell is not None: self.cell.append(data)
    def handle_endtag(self,t):
        if t in ('td','th') and self.cell is not None:
            self.row.append(' '.join(''.join(self.cell).split())); self.cell=None
        elif t=='tr' and self.row is not None:
            self.current.append(self.row); self.row=None
        elif t=='table' and self.current is not None:
            self.tables.append(self.current); self.current=None
for source in metadata['source_files']:
    path=root/source['path']
    assert path.stat().st_size==source['bytes']
    assert hashlib.sha256(path.read_bytes()).hexdigest()==source['sha256']
assert hashlib.sha256(candidate_path.read_bytes()).hexdigest()==metadata['candidate_csv_sha256']
parser=Tables()
parser.feed((root/'data/raw/exposure/tc_commercial_vehicle_safety_2021.html').read_text())
tables=[t for t in parser.tables if any(row and row[0]=='Tractor trailers' for row in t)
        and t[0][1:]==[str(y) for y in range(2015,2022)]]
assert len(tables)==1
source_values=next(r[1:] for r in tables[0] if r[0]=='Tractor trailers')
assert len(candidates)==7
assert len({(r['series_id'],r['year']) for r in candidates})==7
assert [int(r['year']) for r in candidates]==list(range(2015,2022))
for row,value in zip(candidates,source_values):
    assert Decimal(row['vkt_million_km'])==Decimal(value.replace(',',''))>0
    assert Decimal(metadata['figure_28_vkt_million_km_by_year'][row['year']]).quantize(Decimal('0.1'))==Decimal(row['vkt_million_km'])
    assert row['unit']=='million_vehicle_kilometres'
    assert row['estimate_type']=='econometric_forecast'
    assert row['minimum_weight_kg']=='15000' and row['weight_operator']=='>'
    assert row['eligibility_status']=='not_validated_for_ncdb08_rates'
print('All seven annual cells match source Table10; Figure28 rounding, candidate key/domain checks and five source-file hashes pass.')

All seven annual cells match source Table10; Figure28 rounding, candidate key/domain checks and five source-file hashes pass.


In [3]:
# Direct collision query: events and tractor involvements remain separate.
sql = """SELECT C_YEAR AS year, count(*) AS casualty_events,
count(*) FILTER(WHERE C_SEV='1') AS fatal_events,
count(*) FILTER(WHERE C_SEV='2') AS injury_events,
sum(road_tractor_vehicle_count) AS known_tractor_involvements,
count(*) FILTER(WHERE road_tractor_vehicle_count>1) AS multi_tractor_events,
count(*) FILTER(WHERE has_unidentified_tractor) AS unidentified_tractor_events,
count(*) FILTER(WHERE NOT is_analysis_candidate) AS strict_exclusions
FROM read_parquet(?) WHERE is_road_tractor AND C_YEAR BETWEEN 2015 AND 2021
GROUP BY C_YEAR ORDER BY C_YEAR"""
con = duckdb.connect()
raw = con.execute(sql,[str(events_path)]).fetchall()
columns = [d[0] for d in con.description]
annual = [dict(zip(columns,r)) for r in raw]
assert [r['year'] for r in annual]==list(range(2015,2022))
for r in annual:
    assert r['casualty_events']==r['fatal_events']+r['injury_events']
    assert r['known_tractor_involvements']>=r['casualty_events']
    assert r['multi_tractor_events']>0 and r['unidentified_tractor_events']==0
    assert 0<=r['strict_exclusions']<=r['casualty_events']
with (out/'numerator_audit.csv').open('w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=columns);w.writeheader();w.writerows(annual)
display(annual)
print('Seven annual numerator partitions reconcile; vehicle involvements exceed unique events.')

[{'year': 2015,
  'casualty_events': 3075,
  'fatal_events': 194,
  'injury_events': 2881,
  'known_tractor_involvements': 3282,
  'multi_tractor_events': 161,
  'unidentified_tractor_events': 0,
  'strict_exclusions': 0},
 {'year': 2016,
  'casualty_events': 3010,
  'fatal_events': 193,
  'injury_events': 2817,
  'known_tractor_involvements': 3208,
  'multi_tractor_events': 174,
  'unidentified_tractor_events': 0,
  'strict_exclusions': 0},
 {'year': 2017,
  'casualty_events': 3178,
  'fatal_events': 216,
  'injury_events': 2962,
  'known_tractor_involvements': 3445,
  'multi_tractor_events': 208,
  'unidentified_tractor_events': 0,
  'strict_exclusions': 2},
 {'year': 2018,
  'casualty_events': 2990,
  'fatal_events': 190,
  'injury_events': 2800,
  'known_tractor_involvements': 3264,
  'multi_tractor_events': 207,
  'unidentified_tractor_events': 0,
  'strict_exclusions': 0},
 {'year': 2019,
  'casualty_events': 2775,
  'fatal_events': 196,
  'injury_events': 2579,
  'known_tractor_

Seven annual numerator partitions reconcile; vehicle involvements exceed unique events.


In [4]:
# Fingerprint the exact audited inputs and retain the actual SQL.
def sha(path): return hashlib.sha256(path.read_bytes()).hexdigest()
inputs = [events_path,candidate_path,metadata_path,root/'data/sources.json']
summary = {'decision':'not_approved_for_primary_driving_risk_rate',
    'inputs_sha256':{str(p.relative_to(root)):sha(p) for p in inputs},
    'numerator_years':[r['year'] for r in annual],
    'rate_calculated':False,
    'qualification':'Source/weight/geographic/model compatibility is a substantive gate; successful numerical checks do not resolve it.',
    'sql':sql,'sql_parameter':'data/derived/ncdb/collisions.parquet'}
(out/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
con.close()
display(summary)

{'decision': 'not_approved_for_primary_driving_risk_rate',
 'inputs_sha256': {'data/derived/ncdb/collisions.parquet': 'a0417eb0cb5f0da6417fce387eba7d0063362f9fd198a55117330ccd09b29688',
  'data/reference/tractor_exposure_candidates.csv': '3faa5533286a8ac39694d073bc9566ef9d31259daf4fd26235fa44871a78bf99',
  'data/reference/tractor_exposure_candidates.metadata.json': '95eb6f02d64318aabac8e9a6aea8984160d726e69666306b96086309d4fc7148',
  'data/sources.json': 'f4d313eb3a1a82db93446fba7d5e6e6ed8736263f2fdb20c8cf45e78faf309a6'},
 'numerator_years': [2015, 2016, 2017, 2018, 2019, 2020, 2021],
 'rate_calculated': False,
 'qualification': 'Source/weight/geographic/model compatibility is a substantive gate; successful numerical checks do not resolve it.',
 'sql': "SELECT C_YEAR AS year, count(*) AS casualty_events,\ncount(*) FILTER(WHERE C_SEV='1') AS fatal_events,\ncount(*) FILTER(WHERE C_SEV='2') AS injury_events,\nsum(road_tractor_vehicle_count) AS known_tractor_involvements,\ncount(*) FILTER(

## Disposition

The candidate is preserved as modeled exposure context. It is not an approved denominator for a primary observed road-tractor driving-risk trend. A later explicitly model-normalized sensitivity could be considered with its mismatch and uncertainty retained; it would not resolve population compatibility or fill 2022–2025. Event/VKT and vehicle-involvement/VKT are different estimands, both potentially meaningful when the population boundary is defensible. Neither implies fault or a causal effect.